# Semana 3 - Analisis Tecnico

Probamos RSI, MACD, Bandas de Bollinger y soportes/resistencias.
Foco: YPF (papel principal del portfolio) y comparacion con el resto.

In [1]:
import sys
sys.path.append(".")

from src.tecnico import (
    rsi, macd, bollinger,
    soportes_resistencias,
    señales_tecnicas, mostrar_señales,
)
from src.graficos import (
    graficar_rsi, graficar_macd,
    graficar_bollinger, graficar_soportes_resistencias,
    grafico_candlestick,
)
from src.datos import cargar_universo

universo = cargar_universo()
print("Modulos tecnicos cargados")

Modulos tecnicos cargados


## 1. Resumen de senales para YPF

In [2]:
mostrar_señales("YPF", periodo="1y")


  SENALES TECNICAS - YPF (precio $44.49)

RSI(14): 56.97
  -> Zona neutral

MACD: 0.442 | Senal: 0.5465 | Hist: -0.1045
  -> Bajista (MACD<Senal)

Bollinger: inf $40.6 | media $43.03 | sup $45.47
Posicion en banda: 79.8%
  -> Zona media



## 2. RSI - Indice de Fuerza Relativa

**Lectura rapida**:
- RSI > 70: sobrecompra, papel "caro" en lo inmediato
- RSI < 30: sobreventa, papel "barato" en lo inmediato
- RSI entre 30-70: zona neutral

In [3]:
graficar_rsi("YPF", periodo="1y", ventana=14)

In [4]:
rsi_serie = rsi("YPF", ventana=14, periodo="1y")
rsi_serie.tail(10).round(2)

Date
2026-04-28 00:00:00-04:00    56.41
2026-04-29 00:00:00-04:00    62.47
2026-04-30 00:00:00-04:00    63.81
2026-05-01 00:00:00-04:00    53.40
2026-05-04 00:00:00-04:00    57.78
2026-05-05 00:00:00-04:00    55.32
2026-05-06 00:00:00-04:00    51.68
2026-05-07 00:00:00-04:00    49.41
2026-05-08 00:00:00-04:00    47.31
2026-05-11 00:00:00-04:00    56.97
Name: RSI_14, dtype: float64

## 3. MACD - Momentum y cruces

**Lectura rapida**:
- MACD cruza Senal de abajo hacia arriba = senal de compra
- MACD cruza Senal de arriba hacia abajo = senal de venta
- Histograma creciente = momentum acelerando

In [5]:
graficar_macd("YPF", periodo="1y")

In [6]:
macd_df = macd("YPF", periodo="1y")
macd_df.tail(10).round(3)

,MACD,Senal,Histograma
Date,,,
2026-04-28 00:00:00-04:00,0.497,0.613,-0.116
2026-04-29 00:00:00-04:00,0.620,0.615,0.005
2026-04-30 00:00:00-04:00,0.737,0.639,0.098
2026-05-01 00:00:00-04:00,0.678,0.647,0.031
2026-05-04 00:00:00-04:00,0.708,0.659,0.049
2026-05-05 00:00:00-04:00,0.687,0.665,0.022
2026-05-06 00:00:00-04:00,0.606,0.653,-0.047
2026-05-07 00:00:00-04:00,0.499,0.622,-0.123
2026-05-08 00:00:00-04:00,0.375,0.573,-0.198


## 4. Bandas de Bollinger - Volatilidad y mean-reversion

**Lectura rapida**:
- Precio toca banda superior = posible techo de corto plazo
- Precio toca banda inferior = posible piso de corto plazo
- Bandas estrechas = baja volatilidad, suele preceder movimientos fuertes

In [7]:
graficar_bollinger("YPF", periodo="1y", ventana=20, desv=2)

In [8]:
bb = bollinger("YPF", periodo="1y")
bb.tail(5).round(2)

,Close,Media,Banda_Superior,Banda_Inferior,Ancho_%
Date,,,,,
2026-05-05 00:00:00-04:00,43.97,43.02,45.36,40.68,21.77
2026-05-06 00:00:00-04:00,43.26,43.01,45.34,40.67,21.71
2026-05-07 00:00:00-04:00,42.80,43.01,45.34,40.68,21.68
2026-05-08 00:00:00-04:00,42.37,42.98,45.33,40.64,21.86
2026-05-11 00:00:00-04:00,44.44,43.03,45.46,40.60,22.57


## 5. Soportes y Resistencias

Detecta niveles donde el precio rebotó historicamente (max/min locales).
Util para definir stops y targets.

In [9]:
sr = soportes_resistencias("YPF", periodo="6mo", ventana=10, top_n=5)
print(f"Precio actual: ${sr['precio_actual']:.2f}\n")

print("RESISTENCIAS (techos arriba del precio):")
for fecha, nivel in sr["resistencias"]:
    print(f"  ${nivel:.2f}  ({fecha.strftime('%d %b %Y')})")

print("\nSOPORTES (pisos abajo del precio):")
for fecha, nivel in sr["soportes"]:
    print(f"  ${nivel:.2f}  ({fecha.strftime('%d %b %Y')})")

Precio actual: $44.44

RESISTENCIAS (techos arriba del precio):
  $46.22  (31 Mar 2026)

SOPORTES (pisos abajo del precio):
  $40.59  (17 Apr 2026)
  $34.67  (03 Mar 2026)
  $34.06  (17 Dec 2025)
  $32.78  (07 Jan 2026)


In [10]:
graficar_soportes_resistencias("YPF", periodo="6mo", ventana=10, top_n=5)

## 6. Comparacion: senales del portfolio completo

In [11]:
import pandas as pd

mi_portfolio = universo["portfolios"]["personal"]
filas = []
for t in mi_portfolio:
    s = señales_tecnicas(t, periodo="1y")
    if s:
        filas.append({
            "Ticker": s["ticker"],
            "Precio": s["precio_actual"],
            "RSI": s["RSI"],
            "RSI lectura": s["RSI_interpretacion"],
            "MACD lectura": s["MACD_interpretacion"],
            "BB posicion %": s["BB_posicion_%"],
        })

df_senales = pd.DataFrame(filas)
df_senales

,Ticker,Precio,RSI,RSI lectura,MACD lectura,BB posicion %
0,YPF,44.44,56.80,Zona neutral,Bajista (MACD<Senal),79.0
1,PAM,81.08,47.28,Zona neutral,Bajista (MACD<Senal),41.0
2,TGS,29.98,44.03,Zona neutral,Bajista (MACD<Senal),27.5
3,MELI,1553.90,31.18,Zona neutral,"Bajista (MACD<Senal, momentum bajando)",-29.6


## 7. Cross-check: candlestick puro de YPF

Para comparar lo que dicen los indicadores contra el grafico real.

In [12]:
grafico_candlestick("YPF", periodo="6mo")